In [ ]:
!pip install -q bertopic umap-learn hdbscan

In [ ]:
import re
import pandas as pd
import nltk
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
import torch
nltk.download("punkt")
nltk.download("punkt_tab")
import numpy as np
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import (CountVectorizer, ENGLISH_STOP_WORDS)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
OUT_DIR = Path("/content/drive/MyDrive/thesis_results/BERTopic")
OUT_DIR.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Splitting the dataset into sentences**

In [ ]:
SCOTBESS = pd.read_csv("SCOTBESS.csv")

print(SCOTBESS.columns.tolist())
print("Documents:", len(SCOTBESS))


['document_id', 'project', 'filename', 'source', 'final_masked_text']
Documents: 1675


In [ ]:
#splitting the text into sentences as reccomended in the BERTopic documentation
#retaining IDs for tracing to which document the sentence belongs to
sentence_rows = []

for row in SCOTBESS.itertuples(index=False):
    for sentence_id, sentence in enumerate(sent_tokenize(row.final_masked_text)):
        sentence = re.sub(r"\s+", " ", sentence).strip()

        sentence_rows.append({
            "document_id": row.document_id,
            "sentence_id": sentence_id,
            "sentence_text": sentence,
            "project": row.project,
            "filename": row.filename,
            "source": row.source})

sentences_df = pd.DataFrame(sentence_rows)
sentences = sentences_df["sentence_text"].tolist()

print("Documents:", len(SCOTBESS))
print("Extracted sentences:", len(sentences_df))

sentences_df.head()

Documents: 1675
Extracted sentences: 33527


,document_id,sentence_id,sentence_text,project,filename,source
0,0,0,The ground on which the proposed energy site i...,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure
1,0,1,The embankment would structurally safe.,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure
2,0,2,A part of the site is not owned by the propose...,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure
3,1,0,We have serious reservations that there has no...,Aberdeen_City_210665_DPP,extracted_210665_DPP-Objects_-_Ken_Cumming__Ve...,no_structure
4,1,1,From the information supplied the site will ha...,Aberdeen_City_210665_DPP,extracted_210665_DPP-Objects_-_Ken_Cumming__Ve...,no_structure


In [ ]:
sentences_df["word_count"] = (sentences_df["sentence_text"].str.split().str.len())

sentences_df["word_count"].describe()

,word_count
count,33527.000000
mean,22.390461
std,13.627202
min,1.000000
25%,14.000000
50%,21.000000
75%,28.000000
max,418.000000


In [ ]:
#checking very long sentences
with pd.option_context("display.max_colwidth", None,"display.max_rows", 10):
    display(sentences_df.nlargest(10, "word_count")[["document_id","sentence_id","word_count", "sentence_text"]])
    #the long senteces are valid, their lenght results from people not using punctuation

,document_id,sentence_id,word_count,sentence_text
13103,1909,1,418,"There was a visit on Friday 28th of October from [PERSON], the attitude of said person was that regardless of how the residents feel about the impact on the environment, the disruption it will cause to our lives, the fact that we have lived for the past 5 years putting up with the work being done to the already well established power plant on [PLACE] that we were assured would stop by 2020 then covid struck, they then said they needed another year with the work on site to be completed then it would be totally unmanned, this has never happened they are still there working with machinery, and I am assuming that it is the infrastructure for the battery plants, the noise is constant from the sub station there is a buzzing that comes from the overhead pylons that becomes like a toothache you can’t get rid of.I keep horses on the land that borders onto the land that is earmarked for development, I exercise the horses daily on [PLACE] which at times can be quite challenging with the amount of traffic that is on the road just now, if this development goes ahead I feel I will become a prisoner in my own home as I will not be able to negotiate the volume of traffic that the development will cause,my horses being flight animals will also be held captive, they would also need to close the road to get the containers (the size of double decker buses) on site and there is going to be 18 of them with fans on each container running 24-7 to keep them cool which is going to create more noise and the fact they want to put a sonic fence around it to try and lessen the noise impact quite frankly scares the life from me.and all being done without consultation with the residents in the surrounding area .When I purchased my property it was the fact it was on a green belt and would be good for my mental health and the well-being of my animals, was it not the Scottish government that have been telling us all to get healthier and do more exercise get out into the fresh air experience the pleasure’s the countryside has to offer, we will be living with-in an industrial site on our doorstep with constant noise and daily disruptions, I would like there to be more negotiation and reassurance’s before this is( in the words of the planning officer rubber stamped)."
32269,3841,4,361,"So this is going to just be another hurdle we need to cross when we come to it, the standard net zero cart before the horse planning In diagram reference 10109449 S4N [PLACE] Design, the diagram key showed a blue line for a SuDs lines feature, excuse my possible ignorance of understanding this drawing, but this line is showing as directly connecting to the [PLACE], which should there be any run off/contamination from this facility it will flow to this burn which travels through farmland and joins the local river When reviewing the SEPA food risk map (which I have included) the [PLACE] area already shows as being an affected area, and this is prior to this application being constructed or that of West of Orkney’s massive substation which would also be in close proximity to this facility (which has planning in principal already) or [PLACE] BESS up to 300MW (ECU ref - [APPLICATION_CODE]) neither of which I add appear on any drawings/diagrams or visualisations for this application All to which I would imagine cumulatively would make this issue of flood risk increase There’s currently 1 4 GW of BESS in the planning process on the stretch of road between [PLACE]/[PLACE]/[PLACE] (a distance of 2 5 miles at most) impact and risk of these and the other developments is unacceptable As a close neighbour of this site and the many other applications for transmission which are being applied for in this area, I fully believe the individuals who make the decision on this application should have a site visit of the area being made aware of all current applications prior to deciding, to visualise how this area for which the [PLACE] r

In [ ]:
documents = sentences_df["sentence_text"].tolist()

print("Sentences used:", len(documents))

Sentences used: 33527


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedding_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=device)

embeddings = embedding_model.encode(documents, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

print("Embeddings shape:", embeddings.shape)
np.save(OUT_DIR / "SCOTBESS_MPNET_SENTENCE_EMBEDDINGS.npy", embeddings)

Device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1048 [00:00<?, ?it/s]

Embeddings shape: (33527, 768)


**Topic exploration with BERTopic**

In [ ]:
sentences_df = pd.read_csv("SCOTBESS_SENTENCES.csv")
embeddings = np.load("SCOTBESS_MPNET_SENTENCE_EMBEDDINGS.npy")

documents = sentences_df["sentence_text"].tolist()

assert len(documents) == embeddings.shape[0]
print(embeddings.shape)

(33527, 768)


In [ ]:
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
hdbscan_model = HDBSCAN(min_cluster_size=50, min_samples=10, metric="euclidean", cluster_selection_method="eom", prediction_data=True)

placeholder_terms = {"person","place","email","phone","link","application_code","organization"}

custom_stopwords = list(set(ENGLISH_STOP_WORDS).union(placeholder_terms))

vectorizer_model = CountVectorizer(stop_words=custom_stopwords, ngram_range=(1, 2), min_df=5)

ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [ ]:
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    nr_topics=30, # tested several values; reducing to 30 topics produced the most interpretable results, although some noisy topics remain.
    top_n_words=10,
    calculate_probabilities=False,
    verbose=True)

topics, probabilities = topic_model.fit_transform(documents, embeddings)

2026-08-02 17:34:23,070 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-02 17:35:22,623 - BERTopic - Dimensionality - Completed ✓
2026-08-02 17:35:22,625 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-02 17:35:28,824 - BERTopic - Cluster - Completed ✓
2026-08-02 17:35:28,825 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-08-02 17:35:29,820 - BERTopic - Representation - Completed ✓
2026-08-02 17:35:29,822 - BERTopic - Topic reduction - Reducing number of topics
2026-08-02 17:35:29,880 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-02 17:35:30,876 - BERTopic - Representation - Completed ✓
2026-08-02 17:35:30,882 - BERTopic - Topic reduction - Reduced number of topics from 124 to 30


In [ ]:
topic_info = topic_model.get_topic_info()

display(topic_info.head(30))

outlier_count = sum(topic == -1 for topic in topics)

print(f"Topics found: {(topic_info['Topic'] != -1).sum()}")
print(f"Outliers: {outlier_count:,} ({outlier_count / len(topics):.1%})")

,Topic,Count,Name,Representation,Representative_Docs
0,-1,13864,-1_energy_infrastructure_development_public,"[energy, infrastructure, development, public, ...",[I wish to object to planning application [APP...
1,0,3412,0_npf4_eia_policies_national planning,"[npf4, eia, policies, national planning, frame...",[NPF4 Policies 22 and 23 require developments ...
2,1,2125,1_road_traffic_vehicles_roads,"[road, traffic, vehicles, roads, access, const...","[Additionally, the anticipated closure of [PLA..."
3,2,1859,2_battery_storage_batteries_energy storage,"[battery, storage, batteries, energy storage, ...",[Public Health and Safety Threats The [PLACE] ...
4,3,1551,3_noise_background_light pollution_levels,"[noise, background, light pollution, levels, p...","[Additionally, HWLDP Policy 28 stipulates that..."
5,4,1291,4_species_habitat_birds_wildlife,"[species, habitat, birds, wildlife, trees, bat...","[Construction activities, noise pollution, and..."
6,5,1265,5_bess_proposed bess_bess site_bess sites,"[bess, proposed bess, bess site, bess sites, b...",[The [ORGANIZATION] strongly advises against s...
7,6,1246,6_emergency_explosion_safety plan_environmenta...,"[emergency, explosion, safety plan, environmen...",[Given the absence of a comprehensive fire saf...
8,7,1006,7_substation_turbines_metres_lines,"[substation, turbines, metres, lines, substati...",[The construction of multiple 400kV overhead p...
9,8,971,8_landscape visual_cultural_heritage_historical,"[landscape visual, cultural, heritage, histori...",[Cultural Heritage - The [PLACE] The developme...


Topics found: 29
Outliers: 13,864 (41.4%)


In [ ]:
sentences_df["topic"] = topics
sentences_df["topic_probability"] = probabilities

In [ ]:
topic_info = topic_model.get_topic_info()

topic_name_map = dict(zip(topic_info["Topic"], topic_info["Name"]))

sentences_df["topic"] = topics
sentences_df["topic_probability"] = probabilities
sentences_df["topic_name"] = (sentences_df["topic"].map(topic_name_map))
sentences_df.head()

,document_id,sentence_id,sentence_text,project,filename,source,topic,topic_probability,topic_name
0,0,0,The ground on which the proposed energy site i...,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure,1,1.000000,1_road_traffic_vehicles_roads
1,0,1,The embankment would structurally safe.,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure,8,0.707920,8_landscape visual_cultural_heritage_historical
2,0,2,A part of the site is not owned by the propose...,Aberdeen_City_210665_DPP,210665_DPP-Objects-Mr_Marc_Evans_FULL_-2025345...,comment_structure,0,1.000000,0_npf4_eia_policies_national planning
3,1,0,We have serious reservations that there has no...,Aberdeen_City_210665_DPP,extracted_210665_DPP-Objects_-_Ken_Cumming__Ve...,no_structure,2,0.611774,2_battery_storage_batteries_energy storage
4,1,1,From the information supplied the site will ha...,Aberdeen_City_210665_DPP,extracted_210665_DPP-Objects_-_Ken_Cumming__Ve...,no_structure,-1,0.000000,-1_energy_infrastructure_development_public


In [ ]:
display(topic_info.head(30))

outlier_count = (sentences_df["topic"] == -1).sum()

print("Topics:", (topic_info["Topic"] != -1).sum())
print(f"Outliers: {outlier_count:,}, {outlier_count / len(sentences_df):.1%})")

,Topic,Count,Name,Representation,Representative_Docs
0,-1,13864,-1_energy_infrastructure_development_public,"[energy, infrastructure, development, public, ...",[I wish to object to planning application [APP...
1,0,3412,0_npf4_eia_policies_national planning,"[npf4, eia, policies, national planning, frame...",[NPF4 Policies 22 and 23 require developments ...
2,1,2125,1_road_traffic_vehicles_roads,"[road, traffic, vehicles, roads, access, const...","[Additionally, the anticipated closure of [PLA..."
3,2,1859,2_battery_storage_batteries_energy storage,"[battery, storage, batteries, energy storage, ...",[Public Health and Safety Threats The [PLACE] ...
4,3,1551,3_noise_background_light pollution_levels,"[noise, background, light pollution, levels, p...","[Additionally, HWLDP Policy 28 stipulates that..."
5,4,1291,4_species_habitat_birds_wildlife,"[species, habitat, birds, wildlife, trees, bat...","[Construction activities, noise pollution, and..."
6,5,1265,5_bess_proposed bess_bess site_bess sites,"[bess, proposed bess, bess site, bess sites, b...",[The [ORGANIZATION] strongly advises against s...
7,6,1246,6_emergency_explosion_safety plan_environmenta...,"[emergency, explosion, safety plan, environmen...",[Given the absence of a comprehensive fire saf...
8,7,1006,7_substation_turbines_metres_lines,"[substation, turbines, metres, lines, substati...",[The construction of multiple 400kV overhead p...
9,8,971,8_landscape visual_cultural_heritage_historical,"[landscape visual, cultural, heritage, histori...",[Cultural Heritage - The [PLACE] The developme...


Topics: 29
Outliers: 13,864, 41.4%)


In [ ]:
topic_coverage = (sentences_df[sentences_df["topic"] != -1].groupby("topic")
    .agg(
        sentences=("sentence_text", "size"),
        documents=("document_id", "nunique"),
        projects=("project", "nunique"),
        mean_probability=("topic_probability", "mean")).reset_index())

topic_coverage["topic_name"] = (topic_coverage["topic"].map(topic_name_map))
topic_coverage = topic_coverage.sort_values("sentences", ascending=False)

display(topic_coverage.head(30))

,topic,sentences,documents,projects,mean_probability,topic_name
0,0,3412,969,75,0.800487,0_npf4_eia_policies_national planning
1,1,2125,669,74,0.955910,1_road_traffic_vehicles_roads
2,2,1859,837,76,0.919506,2_battery_storage_batteries_energy storage
3,3,1551,488,63,0.934767,3_noise_background_light pollution_levels
4,4,1291,461,61,0.783943,4_species_habitat_birds_wildlife
5,5,1265,480,62,0.878801,5_bess_proposed bess_bess site_bess sites
6,6,1246,593,70,0.814461,6_emergency_explosion_safety plan_environmenta...
7,7,1006,446,64,0.937985,7_substation_turbines_metres_lines
8,8,971,471,68,0.927151,8_landscape visual_cultural_heritage_historical
9,9,915,367,55,0.816592,9_water_flooding_flood_runoff


In [ ]:
for topic_id in topic_info.loc[topic_info["Topic"] != -1, "Topic"].head(30):
    print("\n" + "=" * 80)
    print("TOPIC:", topic_id, topic_name_map[topic_id])

    examples = ( sentences_df[sentences_df["topic"] == topic_id].sort_values("topic_probability", ascending=False).head(10))

    for sentence in examples["sentence_text"]:
        print("-", sentence)


TOPIC: 0 0_npf4_eia_policies_national planning
- This installation along with any other similar proposals needs to be sited well away from dwelling houses.
- Allowing this proposal to go ahead would fundamentally contradict the risk assessments, safety protocols, and development exclusion zones that have been enforced for years to protect critical infrastructure, the environment, and public health.
- A part of the site is not owned by the proposed development and is owned by another party - I am lead to believe the current owner is trying to adopt our ground into the proposed site.
- Policy Breaches 5.
- As such Policy B6 would apply.
- Without this information, it is not possible to determine whether the modelling presented is accurate or reflective of the development’s year-round impact.
- No information has been provided about how local wildlife will be protected or relocated, or whether the project includes any biodiversity assessment or mitigation strategy.
- Many Thanks [PERSON]

In [ ]:
sentences_df.to_csv(OUT_DIR / "SCOTBESS_SENTENCES_WITH_TOPICS.csv", index=False, encoding="utf-8")
topic_info.to_csv(OUT_DIR / "SCOTBESS_BERTOPIC_TOPIC_INFO.csv", index=False, encoding="utf-8",)
topic_coverage.to_csv(OUT_DIR / "SCOTBESS_BERTOPIC_TOPIC_COVERAGE.csv", index=False, encoding="utf-8")

topic_model.save(OUT_DIR / "SCOTBESS_BERTOPIC_MODEL", serialization="safetensors", save_ctfidf=True)